In [42]:
import pandas as pd
pd.set_option('display.max_columns', None)

In [43]:
df = pd.read_csv('../data/trusted/base_numerica_trusted.csv')
df.head(1)

,id,id_player_casa,gols_casa,id_time_casa,id_player_fora,gols_fora,id_time_fora,gols_total,id_resultado
0,1404256,603262,3,21,603265,3,6,6,0


In [50]:
# Casa
jogos_casa = df[['id','id_player_casa','id_time_casa','gols_casa','gols_fora']].copy()
jogos_casa = jogos_casa.rename(columns={
    'id_player_casa':'player_id',
    'id_time_casa':'time_id',
    'gols_casa':'gols_marcados',
    'gols_fora':'gols_sofridos'
})
jogos_casa['resultado'] = jogos_casa.apply(lambda x: 'W' if x['gols_marcados'] > x['gols_sofridos'] 
                                           else 'L' if x['gols_marcados'] < x['gols_sofridos'] else 'D', axis=1)

# Fora
jogos_fora = df[['id','id_player_fora','id_time_fora','gols_fora','gols_casa']].copy()
jogos_fora = jogos_fora.rename(columns={
    'id_player_fora':'player_id',
    'id_time_fora':'time_id',
    'gols_fora':'gols_marcados',
    'gols_casa':'gols_sofridos'
})
jogos_fora['resultado'] = jogos_fora.apply(lambda x: 'W' if x['gols_marcados'] > x['gols_sofridos'] 
                                           else 'L' if x['gols_marcados'] < x['gols_sofridos'] else 'D', axis=1)

# Junta tudo
long_df = pd.concat([jogos_casa, jogos_fora], ignore_index=True)


In [51]:
N = 5

# Média de gols marcados
long_df['media_gols_marcados'] = long_df.groupby('player_id')['gols_marcados'].transform(
    lambda x: x.shift().rolling(N, min_periods=1).mean()
)

# Média de gols sofridos
long_df['media_gols_sofridos'] = long_df.groupby('player_id')['gols_sofridos'].transform(
    lambda x: x.shift().rolling(N, min_periods=1).mean()
)

# Winrate acumulado
long_df['vitoria'] = (long_df['resultado'] == 'W').astype(int)
long_df['winrate'] = long_df.groupby('player_id')['vitoria'].transform(
    lambda x: x.shift().expanding().mean()
)

# Experiência = nº de jogos disputados
long_df['jogos_disputados'] = long_df.groupby('player_id').cumcount()

# Forma recente (últimos 5 jogos)
long_df['forma_recent_vitorias'] = long_df.groupby('player_id')['vitoria'].transform(
    lambda x: x.shift().rolling(5, min_periods=1).sum()
)
long_df['forma_recent_gols'] = long_df.groupby('player_id')['gols_marcados'].transform(
    lambda x: x.shift().rolling(5, min_periods=1).mean()
)

# Sequência de resultados codificada
mapa_res = {'W':1, 'D':0, 'L':-1}
long_df['res_codificado'] = long_df['resultado'].map(mapa_res)
long_df['sequencia_res_5'] = long_df.groupby('player_id')['res_codificado'].transform(
    lambda x: x.shift().rolling(5, min_periods=1).sum()
)

long_df = long_df.fillna(0)


In [54]:
display(long_df)

,id,player_id,time_id,gols_marcados,gols_sofridos,lado
0,0,603262,21,3,3,casa
1,1,603263,20,2,4,casa
2,2,603264,2,4,3,casa
3,3,603265,6,4,6,casa
4,4,603266,9,1,3,casa
...,...,...,...,...,...,...
316819,158407,718276,20,2,2,fora
316820,158408,718277,2,3,1,fora
316821,158409,718278,21,2,4,fora
316822,158410,718146,32,1,4,fora
